# Solutions — UDFs and Pandas UDFs (Hard 09)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql.functions import pandas_udf, udf
from pyspark.sql import Window
import pandas as pd

trips        = spark.table("samples.nyctaxi.trips")
transactions = spark.table("samples.bakehouse.sales_transactions")
customers    = spark.table("samples.bakehouse.sales_customers")


## Solution 1 — Basic Python UDF — Fare Category

In [ ]:
@udf(T.StringType())
def fare_category(amount):
    if amount is None:       return None
    if amount < 5:           return "Short"
    elif amount <= 20:       return "Medium"
    else:                    return "Long"

result_1 = (
    trips
    .withColumn("fare_category", fare_category("fare_amount"))
    .select("tpep_pickup_datetime", "trip_distance", "fare_amount", "fare_category")
)

## Solution 2 — UDF with Multiple Inputs — Formatted Name

In [ ]:
@udf(T.StringType())
def formatted_name(first, last):
    if first is None or last is None:
        return None
    return f"{last.upper()}, {first.title()}"

result_2 = (
    customers
    .withColumn("formatted_name", formatted_name("first_name", "last_name"))
    .select("customerID", "formatted_name", "country")
)

## Solution 3 — UDF for Card Number Masking

In [ ]:
@udf(T.StringType())
def mask_card(card_num):
    if card_num is None:
        return None
    s = str(card_num)
    return "****-****-****-" + s[-4:]

result_3 = (
    transactions
    .withColumn("masked_card", mask_card(F.col("cardNumber").cast("string")))
    .select("transactionID", "cardNumber", "masked_card")
    .limit(100)
)


## Solution 4 — pandas_udf (Scalar) — Z-score of Fare Amount

In [ ]:
@pandas_udf(T.DoubleType())
def zscore_fare(fares: pd.Series) -> pd.Series:
    mean = fares.mean()
    std  = fares.std()
    if std == 0:
        return pd.Series([0.0] * len(fares))
    return (fares - mean) / std

result_4 = (
    trips
    .withColumn("fare_zscore", zscore_fare("fare_amount"))
    .select("fare_amount", "fare_zscore")
)

## Solution 5 — pandas_udf for String Normalisation

In [ ]:
@pandas_udf(T.StringType())
def normalise_product(names: pd.Series) -> pd.Series:
    return names.str.strip().str.title()

result_5 = (
    transactions
    .withColumn("normalised_product_udf", normalise_product("product"))
    .withColumn("normalised_product_builtin", F.initcap(F.trim(F.col("product"))))
    .select("product", "normalised_product_udf", "normalised_product_builtin")
)

## Solution 6 — Register UDF for SQL Use

In [ ]:
def fare_category_func(amount):
    if amount is None:       return None
    if amount < 5:           return "Short"
    elif amount <= 20:       return "Medium"
    else:                    return "Long"

spark.udf.register("fare_category_udf", fare_category_func, T.StringType())

trips.createOrReplaceTempView("trips")

result_6 = spark.sql("""
    SELECT fare_amount, fare_category_udf(fare_amount) AS category
    FROM   trips
""")

## Solution 7 — UDF Performance Comparison — Built-ins Win

In [ ]:
import time

# (a) Python UDF approach - for timing comparison only
@udf(T.DoubleType())
def fare_with_tip_udf(amount):
    return float(amount * 1.1) if amount is not None else None

udf_df = trips.withColumn("fare_with_tip", fare_with_tip_udf("fare_amount"))
t0 = time.time()
udf_df.select("fare_with_tip").count()
udf_time = time.time() - t0
print(f"UDF approach: {udf_time:.2f}s")

# (b) Built-in approach - returned as result_7
builtin_df = trips.withColumn("fare_with_tip", (F.col("fare_amount") * 1.1))
t0 = time.time()
builtin_df.select("fare_with_tip").count()
builtin_time = time.time() - t0
print(f"Built-in approach: {builtin_time:.2f}s")

result_7 = builtin_df.select("fare_amount", "fare_with_tip")

## Solution 8 — Grouped Map with applyInPandas

In [ ]:
def normalise_by_zip(pdf: pd.DataFrame) -> pd.DataFrame:
    lo = pdf["fare_amount"].min()
    hi = pdf["fare_amount"].max()
    pdf["normalised_fare"] = ((pdf["fare_amount"] - lo) / (hi - lo)).round(4) if hi != lo else 0.0
    return pdf[["pickup_zip", "fare_amount", "normalised_fare"]]

schema = "pickup_zip INT, fare_amount DOUBLE, normalised_fare DOUBLE"

result_8 = (
    trips
    .filter(F.col("fare_amount") > 0)
    .groupBy("pickup_zip")
    .applyInPandas(normalise_by_zip, schema=schema)
)
